# Phase 6 — Comparaison des fonctions d’activation

## Objectif

Comparer l'impact de trois fonctions d'activation sur MNIST :
- sigmoid
- tanh
- relu

On garde la même architecture, le même optimizer et les mêmes données.
Seule l’activation change.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import time

In [2]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

X_train = X_train.reshape(-1, 784).astype("float32") / 255.0
X_test = X_test.reshape(-1, 784).astype("float32") / 255.0

print(f"Train : {X_train.shape} | Test : {X_test.shape}")
print(f"Classes uniques : {np.unique(y_train)}")

Train : (60000, 784) | Test : (10000, 784)
Classes uniques : [0 1 2 3 4 5 6 7 8 9]


## 1. Expérience

On teste trois activations sur la même architecture :
- Dense(128, activation)
- Dense(64, activation)
- Dense(10, softmax)

On garde :
- optimizer = Adam
- loss = sparse_categorical_crossentropy
- métrique = accuracy

In [3]:
activations = ["sigmoid", "tanh", "relu"]

results = []
histories = {}

In [4]:
for activation in activations:
    tf.random.set_seed(42)

    model = keras.Sequential([
        keras.layers.Dense(128, activation=activation, input_shape=(784,)),
        keras.layers.Dense(64, activation=activation),
        keras.layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    start = time.time()

    history = model.fit(
        X_train,
        y_train,
        epochs=10,
        batch_size=64,
        validation_split=0.1,
        verbose=0
    )

    train_time = time.time() - start
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    val_loss_final = history.history["val_loss"][-1]
    val_acc_final = history.history["val_accuracy"][-1]

    results.append({
        "activation": activation,
        "val_loss_final": val_loss_final,
        "val_acc_final": val_acc_final,
        "test_accuracy": test_acc,
        "train_time_s": train_time
    })

    histories[activation] = history.history["val_loss"]

    print(
        f"{activation:8s} | "
        f"val_loss={val_loss_final:.4f} | "
        f"val_acc={val_acc_final:.4f} | "
        f"test_acc={test_acc:.4f} | "
        f"time={train_time:.1f}s"
    )

d:\Cours\5ème Année IPSSI - Paris\Machine Learning\deeplearning-j1-pmc-from-scratch-DONOU-Serge\.venv311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


sigmoid  | val_loss=0.0793 | val_acc=0.9760 | test_acc=0.9749 | time=45.3s
tanh     | val_loss=0.0977 | val_acc=0.9733 | test_acc=0.9722 | time=38.2s
relu     | val_loss=0.1114 | val_acc=0.9743 | test_acc=0.9734 | time=35.6s


## 2. Tableau comparatif

In [5]:
print("\n=== TABLEAU COMPARATIF ACTIVATIONS ===")
print(f"{'Activation':10s} | {'Val loss final':14s} | {'Val acc final':14s} | {'Test acc':10s} | {'Temps (s)':10s}")
print("-" * 80)

for r in results:
    print(
        f"{r['activation']:10s} | "
        f"{r['val_loss_final']:.4f} | "
        f"{r['val_acc_final']:.4f} | "
        f"{r['test_accuracy']:.4f} | "
        f"{r['train_time_s']:.0f}"
    )


=== TABLEAU COMPARATIF ACTIVATIONS ===
Activation | Val loss final | Val acc final  | Test acc   | Temps (s) 
--------------------------------------------------------------------------------
sigmoid    | 0.0793 | 0.9760 | 0.9749 | 45
tanh       | 0.0977 | 0.9733 | 0.9722 | 38
relu       | 0.1114 | 0.9743 | 0.9734 | 36


In [6]:
plt.figure(figsize=(10, 5))

for activation, val_losses in histories.items():
    plt.plot(range(1, 11), val_losses, label=activation, linewidth=2)

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Comparaison des activations sur MNIST")
plt.legend()
plt.grid(True, alpha=0.3)

plt.savefig("phase6_activations_curve.png", dpi=100, bbox_inches="tight")
plt.show()

print("\nCourbe sauvegardée : phase6_activations_curve.png")


Courbe sauvegardée : phase6_activations_curve.png


C:\Users\serge\AppData\Local\Temp\ipykernel_3556\1217631961.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Analyse attendue

### Ce qu’on doit observer
- ReLU converge en général plus vite et plus proprement ;
- tanh fonctionne souvent correctement, mais un peu moins efficacement ;
- sigmoid est généralement plus lente et moins performante sur ce type de réseau.

### Interprétation
Sur MNIST avec cette architecture dense :
- ReLU est souvent le meilleur compromis ;
- tanh peut rester correcte ;
- sigmoid souffre plus facilement de saturation.

### Conclusion
Le choix de la fonction d’activation a un impact direct sur :
- la vitesse de convergence ;
- la loss de validation ;
- l’accuracy finale.